# 03-04-01 과대적합 문제와 정규화의 아이디어

**(The Problem of Overfitting & Regularization) — 시험 범위를 이해한 게 아니라 통째로 외운 학생**

출처: 「Zero to 머신러닝 딥러닝 Master」 ([WikiDocs](https://wikidocs.net/book/21464)) — 이 노트북은 책의 코드 블록을 순서대로 담고 있습니다.


### Block 1 — 구현 원리

In [ ]:
import numpy as np

np.random.seed(0)                     # 난수의 출발점 고정 → 매번 같은 결과 (재현성)

# ==================================================================
# 1. 데이터 만들기
#    진짜 관계는 y = 0.5x² + x + 2 (단순한 2차 곡선)
#    훈련 데이터는 일부러 '10개만' → 과대적합이 잘 일어나는 상황
# ==================================================================
def true_function(x):
    """세상의 '진짜 법칙'. 현실에서는 우리가 절대 모르는 함수."""
    return 0.5 * x**2 + x + 2       # ** 는 거듭제곱 연산자

x_train = np.linspace(-3, 3, 10)     # linspace(시작,끝,개수) : -3~3을 10등분
y_train = true_function(x_train) + np.random.randn(10) * 1.5
# randn = 정규분포 난수(평균0,표준편차1) → ×1.5 하면 '측정 오차' 흉내
# 이 잡음이야말로 과대적합 모델이 '외워버리는' 대상이다

x_test  = np.linspace(-3, 3, 50)     # 처음 보는 새 데이터 50개
y_test  = true_function(x_test) + np.random.randn(50) * 1.5

# ==================================================================
# 2. 다항 특성 만들기 : x → [x, x², x³, ..., x⁹]  (03-03-01 복습)
# ==================================================================
DEGREE = 9                             # 일부러 과하게 높은 차수

def make_poly(x, degree):
    """
    1차원 입력을 다항 특성 행렬로 확장한다.

    비유:
    재료 하나(x)를 받아 1제곱, 2제곱, ... 9제곱까지
    아홉 가지로 손질해 도마 위에 나란히 늘어놓는 것.
    """
    # vstack = 세로로 쌓기, .T = 전치(행↔열 뒤집기)
    # range(1, degree+1) = 1,2,...,9  (0제곱은 상수 1이라 제외)
    return np.vstack([x**k for k in range(1, degree + 1)]).T
    # [식 for k in ...] 는 '리스트 컴프리헨션' — for문을 한 줄로 쓰는 문법

P_train = make_poly(x_train, DEGREE)     # (10, 9)
P_test  = make_poly(x_test,  DEGREE)     # (50, 9)

# ==================================================================
# 3. 특성 스케일링 (x⁹은 자릿수가 폭발하므로 필수 — 03-03-01 4단원)
#    ⚠ 평균·표준편차는 '훈련 데이터에서만' 계산해 테스트에 재사용한다
#      (테스트 데이터를 미리 엿보면 '데이터 누수'가 된다)
# ==================================================================
mu = P_train.mean(axis=0)                # axis=0 → 세로 방향 평균 = 열마다 하나씩
sd = P_train.std(axis=0)                 # 열마다 표준편차(퍼진 정도)

def add_bias_and_scale(P):
    """스케일링한 뒤 맨 앞에 1로 채운 열(x0)을 붙인다."""
    scaled = (P - mu) / sd                   # 브로드캐스팅: (m,9)-(9,) 가 행마다 자동 적용
    ones = np.ones((len(P), 1))            # 1로만 채운 (m,1) 열
    return np.hstack([ones, scaled])       # hstack = 옆으로 붙이기 → (m,10)

X_train = add_bias_and_scale(P_train)
X_test  = add_bias_and_scale(P_test)

# ==================================================================
# 4. 정규화된 정규 방정식으로 θ 구하기 (다음 절에서 유도할 공식)
#    θ = (XᵀX + λL)⁻¹ Xᵀy      L은 왼쪽 위만 0인 단위행렬
# ==================================================================
def fit_ridge(X, y, lam):
    """
    정규화된 선형회귀를 한 번에 푼다.

    비유:
    대각선에 λ를 더하는 것은
    '찌그러진 그릇의 바닥을 살짝 들어 올려'
    최저점이 하나로 또렷해지게 만드는 조치와 같다.
    """
    n_col = X.shape[1]                     # 열 개수 = 파라미터 개수
    L = np.eye(n_col)                       # eye(n) = 대각선이 1인 단위행렬
    L[0, 0] = 0                            # ⭐ θ0(절편)에는 벌금을 매기지 않는다
    # solve(A, b)는 Ax=b를 푼다. inv를 만들어 곱하는 것보다 빠르고 안정적
    return np.linalg.solve(X.T @ X + lam * L, X.T @ y)

def mse(X, y, theta):
    """평균제곱오차. 정규화 항은 '성능 평가'에 넣지 않는다."""
    return np.mean((X @ theta - y) ** 2)

# ==================================================================
# 5. λ를 바꿔가며 훈련/테스트 오차와 계수 크기를 기록
# ==================================================================
print(f"{'lambda':>8} | {'훈련 MSE':>9} | {'테스트 MSE':>10} | {'||θ||':>9}")
print("-" * 48)                        # 문자열 * 숫자 = 그 횟수만큼 반복

for lam in [0, 0.01, 0.1, 1, 10, 100, 1000]:
    theta = fit_ridge(X_train, y_train, lam)
    tr = mse(X_train, y_train, theta)
    te = mse(X_test,  y_test,  theta)
    norm = np.linalg.norm(theta[1:])       # [1:] = 절편 빼고, norm = 벡터 길이
    print(f"{lam:>8} | {tr:>9.4f} | {te:>10.4f} | {norm:>9.3f}")
    # {값:>9.4f} = 오른쪽 정렬 9칸, 소수 4자리 → 표처럼 보이게

### Block 2 — 📝 해설

In [ ]:
import numpy as np

def compute_cost_reg(X, y, theta, lam):
    """
    정규화된 선형회귀 비용을 계산한다.

    비유:
    영수증이 두 줄로 찍힌다.
      ① 물건값  = 예측이 얼마나 틀렸는가
      ② 과태료  = 계수가 얼마나 커졌는가
    총액을 최소로 만드는 것이 학습의 목표다.
    """
    m = len(y)                              # 샘플 개수

    # ① 물건값 : 오차 제곱합
    error = X @ theta - y                    # @ 는 행렬곱
    sse = np.sum(error ** 2)

    # ② 과태료 : θ0을 제외한 계수들의 제곱합
    penalty = lam * np.sum(theta[1:] ** 2)
    # theta[1:] 로 첫 원소(절편)를 건너뛴다 ⭐ 가장 흔한 실수 지점

    return (sse + penalty) / (2 * m)